# 02 — Clustering des journées et des foyers

Mode utilisé dans ce support : **smoke**, destiné à vérifier la chaîne complète. Les valeurs commentées proviennent du rapport calculé, jamais de scores inventés.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "outputs" / "reports" / "results_smoke.json"
if not REPORT.exists():
    raise FileNotFoundError("Exécutez d'abord : .venv\\Scripts\\python scripts\\run_smoke.py --mode smoke")
results = json.loads(REPORT.read_text(encoding="utf-8"))
sns.set_theme(style="whitegrid")

## Question étudiée

Les distances entre profils bruts, coordonnées ACP et latents conduisent-elles aux mêmes groupes ? Nous appliquons le même K-means, avec standardisation ajustée sur l'apprentissage :

$$\min_{C_1,\ldots,C_K}\sum_k\sum_{x_i\in C_k}\|x_i-\mu_k\|_2^2.$$

Les centres sont appris sur l'apprentissage ; validation et test sont affectés aux centres existants. Une silhouette dans deux espaces différents ne suffit pas à déclarer un espace « meilleur ».

In [ ]:
cluster=pd.DataFrame(results['clustering']).T
cluster.index.name='espace_k'; display(cluster)
fig,ax=plt.subplots(figsize=(10,3)); ax.bar(cluster.index,cluster.silhouette_test_same_space)
ax.set(ylabel='Silhouette (espace propre)',title='Diagnostic interne — non comparable seul entre espaces')
plt.xticks(rotation=35,ha='right'); plt.show()

In [ ]:
sizes=pd.DataFrame(cluster.cluster_sizes.tolist(),index=cluster.index).fillna(0)
sizes.plot.bar(stacked=True,figsize=(10,3)); plt.ylabel('Journées test'); plt.title('Effectifs des clusters'); plt.show()

In [ ]:
key='autoencoder_k3'; detail=results['clustering'][key]
fig,ax=plt.subplots(figsize=(10,4))
for k,profile in enumerate(detail['profiles']):
    if profile is None: continue
    med=np.array(profile['median']); q25=np.array(profile['q25']); q75=np.array(profile['q75'])
    ax.plot(med,label=f'cluster {k}'); ax.fill_between(np.arange(48),q25,q75,alpha=.15)
ax.set(xlabel='Créneau',ylabel='Valeur (unité source)',title='Médianes et bandes interquartiles — latent AE, K=3'); ax.legend(); plt.show()
display(pd.Series({k:v['stability_ari_train'] for k,v in results['clustering'].items()},name='ARI entre deux initialisations').to_frame())
display(pd.DataFrame(results['household_groups']['heating_counts']).T.fillna(0).astype(int).rename_axis('groupe foyer'))

## Du jour au foyer

Une représentation de foyer s'obtient par ses proportions de journées dans chaque cluster, éventuellement croisées avec saison et type de jour. Cela évite de confondre une journée et un foyer. Les métadonnées servent ensuite à décrire des écarts à la population de référence, pas à baptiser automatiquement un cluster « chauffage électrique » ou « véhicule électrique ».

Limites : ce smoke test ne mesure pas encore la stabilité par bootstrap ; `n_init=10` réduit seulement la sensibilité de K-means à son initialisation. Questions : pourquoi les proportions sont-elles compositionnelles ? Que change le clustering des formes normalisées par rapport aux niveaux ?